# Youth Brain Drain Risk in BRICS Countries — MOORA & COPRAS (Python)

This notebook reproduces, from scratch in Python, the multi-criteria decision-making (MCDM)
analysis originally carried out in Excel. The goal is to rank the five BRICS countries by their
**indirect brain drain risk** for the young population, using six World Bank indicators as proxy
variables.

**Pipeline:** Entropy weighting → MOORA (significance approach) → COPRAS.
All results are validated against the original Excel workbook.

> Data source: World Bank (2023; health expenditure as % of GDP from 2022).

In [1]:
import numpy as np
import pandas as pd

## 1. Decision matrix

Five alternatives (BRICS countries) evaluated on six criteria.

| Code | Criterion | Type |
|------|-----------|------|
| K1 | Employment rate (15+) | Benefit |
| K2 | GDP per capita (PPP) | Benefit |
| K3 | Inflation | Cost |
| K4 | Life expectancy | Benefit |
| K5 | Unemployment rate | Cost |
| K6 | Health expenditure (% of GDP) | Benefit |

In [2]:
countries = ["Brazil", "China", "India", "Russia", "South Africa"]
criteria  = ["K1", "K2", "K3", "K4", "K5", "K6"]

# True = benefit criterion (higher is better), False = cost criterion (lower is better)
is_benefit = pd.Series([True, True, False, True, False, True], index=criteria)

X = pd.DataFrame(
    [
        [57.9150, 21175.6187, 4.5936, 75.8480,  7.9470, 9.1400],
        [62.7560, 25179.1193, 0.2348, 77.9530,  4.6700, 5.3700],
        [52.3690, 10323.4999, 5.6491, 72.0030,  4.1720, 3.3100],
        [59.7920, 44268.7305, 5.8657, 73.2546,  3.0760, 6.9200],
        [39.7420, 15199.7330, 6.0752, 66.1390, 32.0980, 8.7700],
    ],
    index=countries,
    columns=criteria,
)
X

,K1,K2,K3,K4,K5,K6
Brazil,57.915,21175.6187,4.5936,75.8480,7.947,9.14
China,62.756,25179.1193,0.2348,77.9530,4.670,5.37
India,52.369,10323.4999,5.6491,72.0030,4.172,3.31
Russia,59.792,44268.7305,5.8657,73.2546,3.076,6.92
South Africa,39.742,15199.7330,6.0752,66.1390,32.098,8.77


## 2. Entropy weighting

Criterion weights are derived **objectively** from the data with Shannon entropy, so they do not
depend on subjective judgment. Criteria that differentiate the alternatives more strongly receive
higher weights. The same weights are used for both MOORA and COPRAS to keep the two methods
consistent.

For each criterion *j*: normalize by the column sum to get \(p_{ij}\), compute entropy
\(e_j = -k \sum_i p_{ij}\ln p_{ij}\) with \(k = 1/\ln m\), then the degree of differentiation
\(d_j = 1 - e_j\), and finally \(w_j = d_j / \sum d_j\).

In [3]:
def entropy_weights(df: pd.DataFrame) -> pd.DataFrame:
    """Objective criterion weights via the Shannon entropy method."""
    m = len(df)
    P = df / df.sum(axis=0)                 # column-sum normalization
    k = 1 / np.log(m)
    # use np.where so that 0 * ln(0) is treated as 0
    plnp = np.where(P > 0, P * np.log(P), 0.0)
    e = -k * plnp.sum(axis=0)               # entropy of each criterion
    d = 1 - e                               # degree of differentiation
    w = d / d.sum()                         # normalized weights
    return pd.DataFrame({"e_j": e, "d_j": d, "w_j": w}, index=df.columns)

ew = entropy_weights(X)
w = ew["w_j"]
ew.round(4)

,e_j,d_j,w_j
K1,0.9927,0.0073,0.0145
K2,0.9262,0.0738,0.1463
K3,0.8851,0.1149,0.2278
K4,0.9990,0.0010,0.0019
K5,0.7277,0.2723,0.5400
K6,0.9649,0.0351,0.0695


Unemployment (**K5**) carries by far the highest weight (0.54): it is the criterion that
separates the countries most (South Africa's ~32% unemployment is a strong outlier). Life
expectancy (**K4**) is almost flat across the group, so it gets a near-zero weight.

## 3. MOORA (significance / weighted approach)

The decision matrix is normalized with the **vector norm**
\(x^*_{ij} = x_{ij} / \sqrt{\sum_i x_{ij}^2}\), multiplied by the entropy weights, and for each
country the cost-weighted sum is subtracted from the benefit-weighted sum:
\(y_i = \sum_{benefit} w_j x^*_{ij} - \sum_{cost} w_j x^*_{ij}\).
A higher \(y_i\) means **lower** brain drain risk.

In [4]:
def moora(df: pd.DataFrame, weights: pd.Series, benefit_mask: pd.Series) -> pd.DataFrame:
    """MOORA ratio system with significance coefficients (weights)."""
    norm = df / np.sqrt((df ** 2).sum(axis=0))      # vector normalization
    weighted = norm * weights
    Yi = (weighted.loc[:, benefit_mask].sum(axis=1)
          - weighted.loc[:, ~benefit_mask].sum(axis=1))
    res = pd.DataFrame({"Yi": Yi})
    res["Rank"] = res["Yi"].rank(ascending=False).astype(int)
    return res.sort_values("Rank")

moora_res = moora(X, w, is_benefit)
moora_res.round(4)

,Yi,Rank
China,0.0159,1
Russia,-0.0191,2
Brazil,-0.1195,3
India,-0.1345,4
South Africa,-0.5546,5


## 4. COPRAS (Complex Proportional Assessment)

The matrix is normalized by **column sums**, weighted with the entropy weights, then split into a
benefit sum \(S_{+i}\) and a cost sum \(S_{-i}\). The relative significance is
\(Q_i = S_{+i} + \dfrac{\sum_i S_{-i}}{S_{-i}}\), and the utility degree is
\(N_i = (Q_i / Q_{\max}) \times 100\). The country with \(N_i = 100\) is the best (lowest-risk)
alternative.

*Note:* this matches the simplified \(Q_i\) form used in the original study. The standard
textbook COPRAS adds an \(S_{-\min}\) normalization term; since both are of the form
\(S_{+i} + c/S_{-i}\), they produce the **same ranking** (only the \(N_i\) magnitudes differ).

In [5]:
def copras(df: pd.DataFrame, weights: pd.Series, benefit_mask: pd.Series) -> pd.DataFrame:
    """COPRAS - Complex Proportional Assessment."""
    norm = df / df.sum(axis=0)                       # column-sum normalization
    weighted = norm * weights
    S_plus  = weighted.loc[:, benefit_mask].sum(axis=1)
    S_minus = weighted.loc[:, ~benefit_mask].sum(axis=1)
    Q = S_plus + S_minus.sum() / S_minus             # relative significance
    N = Q / Q.max() * 100                             # utility degree (%)
    res = pd.DataFrame({"S+": S_plus, "S-": S_minus, "Q": Q, "N (%)": N})
    res["Rank"] = res["N (%)"].rank(ascending=False).astype(int)
    return res.sort_values("Rank")

copras_res = copras(X, w, is_benefit)
copras_res.round(4)

,S+,S-,Q,N (%),Rank
China,0.0466,0.0509,15.1263,100.0000,1
Russia,0.0737,0.0916,8.4585,55.9194,2
India,0.0230,0.1008,7.6432,50.5294,3
Brazil,0.0491,0.1293,5.9890,39.5931,4
South Africa,0.0398,0.3953,1.9822,13.1040,5


## 5. Results and conclusion

In [6]:
comparison = pd.DataFrame({
    "MOORA rank":  moora_res["Rank"],
    "COPRAS rank": copras_res["Rank"],
}).sort_values("MOORA rank")
comparison

,MOORA rank,COPRAS rank
China,1,1
Russia,2,2
Brazil,3,4
India,4,3
South Africa,5,5


Both methods agree at the decisive extremes:

- **China** has the **lowest** indirect brain drain risk (best indicators overall).
- **South Africa** has the **highest** risk, driven mainly by its very high unemployment.

The middle of the ranking differs slightly (Brazil and India swap between the two methods), but the
agreement at the extremes — produced by two independent MCDM methods — supports the robustness of
the result. These values reflect **relative, indirect** risk based on proxy indicators, not direct
brain drain figures.

All numbers above reproduce the original Excel workbook exactly.